# Data Exploration - Health Insurance Documents

This notebook explores the health insurance dataset used for the RAG system.

In [ ]:
import sys
sys.path.append('../src')

import json
from pathlib import Path
import pandas as pd
from data_processing import HealthInsuranceDataProcessor

## 1. Load and Process Documents

In [ ]:
# Initialize processor
processor = HealthInsuranceDataProcessor(
    raw_data_path='../data/raw',
    processed_data_path='../data/processed'
)

# Create sample data if not exists
processor.create_sample_data()

# Process documents
chunks = processor.process_documents()

## 2. Dataset Statistics

In [ ]:
# Get statistics
stats = processor.get_statistics()

print(f"Total Chunks: {stats['total_chunks']}")
print(f"Unique Sources: {stats['unique_sources']}")
print(f"Average Chunk Length: {stats['avg_chunk_length']:.0f} characters")
print(f"\nDocuments:")
for source in stats['sources']:
    print(f"  - {source}")

## 3. Chunk Length Distribution

In [ ]:
# Analyze chunk lengths
chunk_lengths = [len(chunk['text']) for chunk in chunks]

df = pd.DataFrame({
    'chunk_length': chunk_lengths,
    'source': [chunk['source'] for chunk in chunks]
})

print(df.describe())

In [ ]:
# Plot distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(chunk_lengths, bins=20, edgecolor='black')
plt.xlabel('Chunk Length (characters)')
plt.ylabel('Frequency')
plt.title('Distribution of Chunk Lengths')
plt.axvline(x=stats['avg_chunk_length'], color='r', linestyle='--', label=f'Mean: {stats["avg_chunk_length"]:.0f}')
plt.legend()
plt.show()

## 4. Chunks by Source

In [ ]:
# Count chunks per source
chunks_by_source = df.groupby('source').size().reset_index(name='count')
chunks_by_source = chunks_by_source.sort_values('count', ascending=False)

print(chunks_by_source)

# Plot
plt.figure(figsize=(12, 6))
plt.bar(range(len(chunks_by_source)), chunks_by_source['count'])
plt.xticks(range(len(chunks_by_source)), chunks_by_source['source'], rotation=45, ha='right')
plt.xlabel('Source Document')
plt.ylabel('Number of Chunks')
plt.title('Chunks per Source Document')
plt.tight_layout()
plt.show()

## 5. Sample Chunks

In [ ]:
# Display sample chunks
print("Sample Chunks:\n")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n{'='*60}")
    print(f"Chunk {i+1}")
    print(f"{'='*60}")
    print(f"Source: {chunk['source']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Length: {len(chunk['text'])} characters")
    print(f"\nText:\n{chunk['text'][:300]}...")

## 6. Word Frequency Analysis

In [ ]:
from collections import Counter
import re

# Combine all text
all_text = ' '.join([chunk['text'] for chunk in chunks]).lower()

# Extract words
words = re.findall(r'\b[a-z]{4,}\b', all_text)

# Count frequency
word_freq = Counter(words).most_common(20)

# Display
print("Top 20 Most Common Words (4+ letters):\n")
for word, count in word_freq:
    print(f"{word:20s}: {count:3d}")

In [ ]:
# Plot word frequency
words_list = [w[0] for w in word_freq]
counts_list = [w[1] for w in word_freq]

plt.figure(figsize=(12, 6))
plt.bar(range(len(words_list)), counts_list)
plt.xticks(range(len(words_list)), words_list, rotation=45, ha='right')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.title('Top 20 Most Common Words')
plt.tight_layout()
plt.show()

## 7. Key Terms Analysis

In [ ]:
# Define health insurance key terms
key_terms = [
    'deductible', 'premium', 'copay', 'coinsurance', 'coverage',
    'claim', 'network', 'provider', 'insurance', 'health',
    'mental', 'preventive', 'emergency', 'prescription'
]

# Count occurrences
term_counts = {term: all_text.count(term) for term in key_terms}
term_counts = dict(sorted(term_counts.items(), key=lambda x: x[1], reverse=True))

print("Key Insurance Terms Frequency:\n")
for term, count in term_counts.items():
    print(f"{term:20s}: {count:3d}")

# Plot
plt.figure(figsize=(12, 6))
plt.bar(range(len(term_counts)), list(term_counts.values()))
plt.xticks(range(len(term_counts)), list(term_counts.keys()), rotation=45, ha='right')
plt.xlabel('Term')
plt.ylabel('Frequency')
plt.title('Health Insurance Key Terms Frequency')
plt.tight_layout()
plt.show()

## Conclusions

- Dataset consists of 5 health insurance documents
- Chunked into ~45 segments for optimal retrieval
- Average chunk size around 500 characters
- Good coverage of key insurance terms
- Balanced distribution across topics